In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


# Comment Classification using Machine Learning

## Project Overview

This project classifies online comments into different categories of toxic behavior using Natural Language Processing (NLP) and Machine Learning.

The model predicts six categories:

- Toxic
- Severe Toxic
- Obscene
- Threat
- Insult
- Identity Hate

The project uses TF-IDF features from both words and characters and a Logistic Regression model for multi-label classification.

## Import the libraries

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import roc_auc_score, classification_report
from scipy.sparse import hstack

## 1. Load Dataset

In [3]:
train_path = "/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip"

df = pd.read_csv(train_path)

print("Dataset Shape:", df.shape)
print("\nColumns:")
print(df.columns)

Dataset Shape: (159571, 8)

Columns:
Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')


## 2. Exploratory Data Analysis

In [4]:
print("Missing values:")
print(df.isnull().sum())

Missing values:
id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64


In [5]:
print("Duplicate comments:", df["comment_text"].duplicated().sum())

Duplicate comments: 0


In [6]:
target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

print("Label counts:")
print(df[target_columns].sum()) 

Label counts:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


This will show the number of positive examples for each category.

## 3. Text Preprocessing

The comment text is cleaned before feature extraction. The preprocessing includes:

- Converting text to lowercase
- Removing URLs
- Removing HTML tags
- Removing IP addresses
- Removing extra spaces

In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\b(?:\d{1,3}\.){3}\d{1,3}\b", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df["clean_text"] = df["comment_text"].apply(clean_text)

In [8]:
print("Original:")
print(df["comment_text"].iloc[0])

print("\nCleaned:")
print(df["clean_text"].iloc[0])

Original:
Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27

Cleaned:
explanation why the edits made under my username hardcore metallica fan were reverted? they weren't vandalisms, just closure on some gas after i voted at new york dolls fac. and please don't remove the template from the talk page since i'm retired now.


## 4. Train-Test Split

In [9]:
X = df["clean_text"]
y = df[target_columns]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (127656,)
Testing data: (31915,)


## 5. Word-Level TF-IDF

TF-IDF converts text into numerical features based on the importance of words and word combinations.

We use:

- Maximum 50,000 features
- Unigrams and bigrams

In [10]:
word_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2)
)

X_train_word = word_vectorizer.fit_transform(X_train)
X_test_word = word_vectorizer.transform(X_test)

print("Training word features:", X_train_word.shape)
print("Testing word features:", X_test_word.shape)

Training word features: (127656, 50000)
Testing word features: (31915, 50000)


## 6. Character-Level TF-IDF

Character-level TF-IDF captures patterns at the character level.

We use character n-grams of size 3 to 5 with a maximum of 50,000 features.

In [11]:
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=50000
)

X_train_char = char_vectorizer.fit_transform(X_train)
X_test_char = char_vectorizer.transform(X_test)

print("Training character features:", X_train_char.shape)
print("Testing character features:", X_test_char.shape)

Training character features: (127656, 50000)
Testing character features: (31915, 50000)


## 7. Combine Word and Character Features

Word-level and character-level TF-IDF features are combined to provide richer text representations for the classification model.

In [12]:
X_train_combined = hstack([X_train_word, X_train_char])
X_test_combined = hstack([X_test_word, X_test_char])

print("Combined training features:", X_train_combined.shape)
print("Combined testing features:", X_test_combined.shape)

Combined training features: (127656, 100000)
Combined testing features: (31915, 100000)


## 8. Train the Classification Model

Model 1: Logistic Regression

A One-vs-Rest Logistic Regression model is used for multi-label classification.

Class weights are balanced to handle the imbalance between the different toxicity categories.

In [13]:
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
)

model.fit(X_train_combined, y_train)

print("Model training completed.")

Model training completed.


## 9. Model Evaluation

The trained model generates probability scores for each of the six toxicity categories. These probabilities are used to evaluate the model using ROC-AUC.

In [14]:
test_scores = model.predict_proba(X_test_combined)

print("Prediction shape:", test_scores.shape)

Prediction shape: (31915, 6)


## ROC-AUC Evaluation

ROC-AUC measures how well the model ranks positive examples higher than negative examples.

Since this is a multi-label problem, we calculate the ROC-AUC separately for each category and then calculate the macro average.

In [15]:
macro_auc = roc_auc_score(
    y_test,
    test_scores,
    average="macro"
)

print("Macro ROC-AUC:", macro_auc)

Macro ROC-AUC: 0.9856075603042501


## ROC-AUC by Category

In [16]:
for i, category in enumerate(target_columns):
    auc = roc_auc_score(y_test.iloc[:, i], test_scores[:, i])
    print(f"{category}: {auc:.4f}")

toxic: 0.9777
severe_toxic: 0.9863
obscene: 0.9905
threat: 0.9949
insult: 0.9829
identity_hate: 0.9813


## Model 2: Linear Support Vector Machine (SVM)

A Linear Support Vector Machine is another classification algorithm suitable for high-dimensional text data.

The model is trained using the same combined word and character TF-IDF features used for Logistic Regression so that the models can be compared fairly.

In [17]:
svm_model = OneVsRestClassifier(
    LinearSVC(
        class_weight="balanced",
        max_iter=2000
    )
)

svm_model.fit(X_train_combined, y_train)

print("SVM training completed.")

SVM training completed.


## Evaluating Linear SVM

The trained Linear SVM is evaluated on the same validation dataset used for Logistic Regression.

Since LinearSVC does not provide probability estimates by default, its decision scores are used to calculate the ROC-AUC.

In [18]:
svm_scores = svm_model.decision_function(X_test_combined)

print("SVM score shape:", svm_scores.shape)

SVM score shape: (31915, 6)


In [19]:
svm_macro_auc = roc_auc_score(
    y_test,
    svm_scores,
    average="macro"
)

print("SVM Macro ROC-AUC:", svm_macro_auc)

SVM Macro ROC-AUC: 0.9742535534991456


## Model 3: Multinomial Naive Bayes

Multinomial Naive Bayes is a probabilistic machine learning algorithm commonly used for text classification.

The model will be trained using the same training and validation split and the same combined TF-IDF features used for the other models.

In [20]:
nb_model = OneVsRestClassifier(
    MultinomialNB()
)

nb_model.fit(X_train_combined, y_train)

print("Naive Bayes training completed.")

Naive Bayes training completed.


In [21]:
nb_scores = nb_model.predict_proba(X_test_combined)

print("Naive Bayes score shape:", nb_scores.shape)

Naive Bayes score shape: (31915, 6)


In [22]:
nb_macro_auc = roc_auc_score(
    y_test,
    nb_scores,
    average="macro"
)

print("Naive Bayes Macro ROC-AUC:", nb_macro_auc)

Naive Bayes Macro ROC-AUC: 0.9282350570311598


## Model Comparison

Three machine learning algorithms were implemented and evaluated using the same validation dataset and the same Macro ROC-AUC metric.

| Model | Macro ROC-AUC |
|---|---:|
| Logistic Regression | 0.98561 |
| Linear SVM | 0.97425 |
| Multinomial Naive Bayes | 0.92824 |

The validation results show different performance across the three algorithms. Logistic Regression achieved the highest Macro ROC-AUC among the three models on the validation set.

Therefore, Logistic Regression was selected for the final training and Kaggle prediction stage.

In [23]:
model_results = {
    "Logistic Regression": 0.98561,
    "Linear SVM": 0.97425,
    "Multinomial Naive Bayes": 0.92824
}

print(model_results)

{'Logistic Regression': 0.98561, 'Linear SVM': 0.97425, 'Multinomial Naive Bayes': 0.92824}


## 10. Load Kaggle Test Data

In [24]:
test_path = "/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip"
kaggle_test = pd.read_csv(test_path)

print("Kaggle test shape:", kaggle_test.shape)
print(kaggle_test.columns)

Kaggle test shape: (153164, 2)
Index(['id', 'comment_text'], dtype='object')


## Preprocess Test Data

In [25]:
kaggle_test["clean_text"] = kaggle_test["comment_text"].apply(clean_text)

print(kaggle_test[["comment_text", "clean_text"]].head())

                                        comment_text  \
0  Yo bitch Ja Rule is more succesful then you'll...   
1  == From RfC == \n\n The title is fine as it is...   
2  " \n\n == Sources == \n\n * Zawe Ashton on Lap...   
3  :If you have a look back at the source, the in...   
4          I don't anonymously edit articles at all.   

                                          clean_text  
0  yo bitch ja rule is more succesful then you'll...  
1    == from rfc == the title is fine as it is, imo.  
2     " == sources == * zawe ashton on lapland — / "  
3  :if you have a look back at the source, the in...  
4          i don't anonymously edit articles at all.  


## 11. Final Model Training on Full Dataset

In [26]:
final_word_vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2)
)

X_full_word = final_word_vectorizer.fit_transform(df["clean_text"])

print("Full training word features:", X_full_word.shape)

Full training word features: (159571, 50000)


In [27]:
final_char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=50000
)

X_full_char = final_char_vectorizer.fit_transform(df["clean_text"])

print("Full training character features:", X_full_char.shape)

Full training character features: (159571, 50000)


In [28]:
X_full_combined = hstack([X_full_word, X_full_char])

print("Full training combined features:", X_full_combined.shape)

Full training combined features: (159571, 100000)


## 12. Transforming Kaggle Test Data

The same TF-IDF vectorizers trained on the complete training dataset are used to transform the Kaggle test data.

The test data is only transformed using the learned vocabulary and IDF values. The vectorizers are not fitted again to avoid data leakage.

In [29]:
X_kaggle_word = final_word_vectorizer.transform(
    kaggle_test["clean_text"]
)

X_kaggle_char = final_char_vectorizer.transform(
    kaggle_test["clean_text"]
)

X_kaggle_combined = hstack([
    X_kaggle_word,
    X_kaggle_char
])

print("Kaggle test word features:", X_kaggle_word.shape)
print("Kaggle test character features:", X_kaggle_char.shape)
print("Kaggle test combined features:", X_kaggle_combined.shape)

Kaggle test word features: (153164, 50000)
Kaggle test character features: (153164, 50000)
Kaggle test combined features: (153164, 100000)


## 13. Training the Final Model

The final Logistic Regression model is trained using the complete labeled training dataset.

Unlike the validation model, which was trained on 80% of the data, this final model uses all 159,571 labeled comments so that it can learn from the maximum available training data before making predictions on the Kaggle test set.

In [30]:
final_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    )
)

final_model.fit(X_full_combined, y)

print("Final model training completed.")

Final model training completed.


## 14. Generating Predictions

The final trained model is used to predict the probability of each toxic category for every comment in the Kaggle test dataset.

Since the Kaggle competition uses ROC-AUC, we will use the predicted probabilities rather than converting them into 0/1 labels.

In [31]:
kaggle_predictions = final_model.predict_proba(
    X_kaggle_combined
)

print("Prediction shape:", kaggle_predictions.shape)

Prediction shape: (153164, 6)


## 15. Creating Kaggle Submission File

The predicted probabilities are organized into the required submission format.

The submission file contains the comment ID and the predicted probability for each of the six toxic categories.

In [32]:
submission = pd.DataFrame(
    kaggle_predictions,
    columns=target_columns
)

submission.insert(0, "id", kaggle_test["id"])

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)

                 id     toxic  severe_toxic   obscene    threat    insult  \
0  00001cee341fdb12  0.999999      0.953719  0.999999  0.697684  0.999652   
1  0000247867823ef7  0.010122      0.003735  0.003149  0.000382  0.011341   
2  00013b17ad220c46  0.045633      0.007073  0.094390  0.000856  0.016935   
3  00017563c3f7919a  0.005851      0.007154  0.004027  0.001021  0.004366   
4  00017695ad8997eb  0.043713      0.007080  0.018967  0.001568  0.008988   

   identity_hate  
0       0.988503  
1       0.007031  
2       0.007385  
3       0.000853  
4       0.001388  
Submission shape: (153164, 7)


## Final Results

The final Logistic Regression model was selected based on the validation results.

### Validation Performance

- Logistic Regression: **0.98561 Macro ROC-AUC**
- Linear SVM: **0.97425 Macro ROC-AUC**
- Multinomial Naive Bayes: **0.92824 Macro ROC-AUC**

The selected Logistic Regression model was retrained using all **159,571 labeled training comments** and used to generate predictions for the Kaggle test dataset.

### Kaggle Performance

The final submission achieved a **Private Leaderboard ROC-AUC of 0.97931**.

The Kaggle submission contained probability scores for all six toxicity categories rather than binary predictions.

i should add this section after comparision markdown , or in last

## Conclusion

This project successfully developed a multi-label toxic comment classification system using NLP and machine learning.

Word and character TF-IDF features were combined, and three models were evaluated. Logistic Regression achieved the highest validation Macro ROC-AUC of **0.98561** and was used for the final Kaggle submission, achieving a **Private Leaderboard ROC-AUC of 0.97931**.